# 3. Status query

Read a fresh controller status and distinguish electrical feedback from optical observation.


## Before you start

Follow [the notebook setup and safety guide](README.md), then select the
project's Python 3.12 kernel. Run cells from top to bottom. Each notebook
is independent. Default execution uses a fictional config and makes no
hardware calls; a skipped operation is **not a successful hardware test**.

For an authorized live session, set `CONFIG` to `ROOT / 'shutters.local.toml'`,
set `TARGET` to the reviewed exact device name, and only then enable the
relevant gate. Keep the source independently protected. Release any GUI,
CLI or other notebook owner first. Restart the kernel after changing setup.


In [ ]:
from pathlib import Path
from dataclasses import asdict
from pprint import pprint
from thorlabs_shutter_control import ShutterHub, load_config

# Works from the repository root or docs/examples. No device access.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'config/shutters.example.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project checkout.')
CONFIG = ROOT / 'config/shutters.example.toml'  # Fictional; offline only.
TARGET = 'example'  # Exact configured name; never select the first device.
RUN_HARDWARE = False

configs = load_config(CONFIG)
selected = next((c for c in configs if c.name == TARGET), None)
if selected is None:
    raise ValueError('TARGET must match an exact configured name.')
print('Selected:', selected.name, selected.serial_number, selected.route.kind)


## 1. Query once, without polling

Connect first: a disconnected `get_status()` returns host metadata rather
than a device measurement. The example uses one explicit query and releases
the owner even if a query fails.


In [ ]:
if RUN_HARDWARE:
    hub = ShutterHub(configs)
    try:
        hub.connect(TARGET)
        status = hub.get_status(TARGET)
        pprint(asdict(status))
        print('Mode:', status.feedback.operating_mode)
        print('Electrical:', status.feedback.electrical_state)
        print('Electrical timestamp (UTC):', status.feedback.electrical_observed_at)
        print('Optical:', status.feedback.shutter_state)
    finally:
        hub.close()
else:
    print('Query skipped; no live status available.')


## 2. Interpret the result

| Field | Meaning |
| --- | --- |
| `requested_state` | Last software request, possibly absent; not feedback |
| `feedback.electrical_state` | Controller-reported `on` / `off` |
| `feedback.operating_mode` | Controller-reported mode |
| `feedback.interlock_enabled` | Configured interlock requirement; not contact continuity |
| `feedback.shutter_state` | Optical state; remains `unknown` without a live independent sensor |
| `transition_confirmed` | Remains false; an electrical reply cannot confirm blade motion |
| `feedback_freshness` | Reply received after request; APT has no transaction ID |

Key/channel/status-bit fields can be unknown. Do not replace unknown with
false or interpret an old optical observation as a continuing sensor.
A timeout invalidates reliable session reuse. Preserve the exception and
any cleanup notes, independently protect the source, and follow
[protected recovery](../python-api.md); never turn ON to test a failed link.
